In [1]:
import torch
print('GPU:', torch.cuda.is_available())
print('GPU Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

GPU: True
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
!pip install transformers accelerate datasets evaluate -q

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset, DatasetDict
from evaluate import load
import torch
import warnings
warnings.filterwarnings('ignore')

print('GPU:', torch.cuda.is_available())
print('GPU Name:', torch.cuda.get_device_name(0))

df = pd.read_csv('../data/cleaned_data.csv')
df = df.dropna(subset=['text_clean'])
df['text_clean'] = df['text_clean'].astype(str)
print('Data loaded:', df.shape)

GPU: True
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
Data loaded: (51055, 4)


In [4]:
labels = df['label'].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label2id)
df['label_id'] = df['label'].map(label2id)
print('Labels:', label2id)

Labels: {'Anxiety': 0, 'Normal': 1, 'Depression': 2, 'Suicidal': 3, 'Stress': 4, 'Bipolar': 5, 'Personality disorder': 6}


In [5]:
train_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df['label'])

def make_dataset(data):
    return Dataset.from_pandas(
        data[['text_clean', 'label_id']].rename(
            columns={'text_clean':'text', 'label_id':'label'}),
        preserve_index=False)

dataset = DatasetDict({
    'train': make_dataset(train_df),
    'validation': make_dataset(val_df),
    'test': make_dataset(test_df)
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 35738
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 7658
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7659
    })
})


In [6]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)
print('Tokenization done!')

Map:   0%|          | 0/35738 [00:00<?, ? examples/s]

Map:   0%|          | 0/7658 [00:00<?, ? examples/s]

Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

Tokenization done!


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    id2label=id2label, label2id=label2id)
print('Model loaded! Parameters:', sum(p.numel() for p in model.parameters()))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded! Parameters: 109487623


In [8]:
training_args = TrainingArguments(
    output_dir='../models/saved/bert_burnout',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
)

metric = load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print('Starting training on RTX 4060!')
trainer.train()
print('Training complete!')

Starting training on RTX 4060!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.531635,0.514069,0.793549
2,0.422970,0.489945,0.800601
3,0.322719,0.481332,0.814965


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training complete!


In [9]:
trainer.save_model('../models/saved/bert_burnout')
tokenizer.save_pretrained('../models/saved/bert_burnout')

results = trainer.evaluate(tokenized_dataset['test'])
print('Test results:', results)
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: on_train_begin must be called before on_evaluate

In [10]:
results = trainer.evaluate(tokenized_dataset['test'])
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")

RuntimeError: on_train_begin must be called before on_evaluate

In [11]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='../models/saved/bert_burnout',
    tokenizer='../models/saved/bert_burnout',
    device=0
)

# Test it!
texts = [
    "I feel so exhausted and hopeless about my studies",
    "I am doing great and feeling motivated today!",
    "I can't focus anymore, everything feels pointless",
    "I'm so stressed about my exams I can't sleep"
]

for text in texts:
    result = classifier(text)
    print(f"Text: {text[:50]}...")
    print(f"Prediction: {result}\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: I feel so exhausted and hopeless about my studies...
Prediction: [{'label': 'Normal', 'score': 0.7722838521003723}]

Text: I am doing great and feeling motivated today!...
Prediction: [{'label': 'Normal', 'score': 0.9894199371337891}]

Text: I can't focus anymore, everything feels pointless...
Prediction: [{'label': 'Suicidal', 'score': 0.5048556327819824}]

Text: I'm so stressed about my exams I can't sleep...
Prediction: [{'label': 'Normal', 'score': 0.9926102757453918}]

